In [ ]:
import numpy as np
import pandas as pd
from scipy.io import loadmat

import pandas as pd
import time
import os
import sys
import zarr
import napari 
import dask.array as da 

pythonPackagePath = os.path.abspath(r'D:\LLSM-CME-ANALYSIS\Final\src')
sys.path.append(pythonPackagePath)
# from parallel import Detector
from gaussian_visualization import visualize_3D_gaussians

In [ ]:
import_trackability = True  # Set to True if you have trackability data

base_dir =  r'Z:\Abhi\LLSM_Analysis'
# Define the file directory and name
# input_file_directory = 'controlOS_analysis/tracking_sweep_results/msn_gap3_rad1/'
input_file_directory = '41OS_0-5min_analysis/tracks/'
input_file_directory_trackability = 'controlOS_analysis/dynROI_tracking_trackability/msn_gap1_rad3/roi_003/'

In [ ]:
# Load the tracks
if import_trackability:
    path_to_tracks = os.path.join(base_dir, input_file_directory_trackability) + 'trackInROI/Channel_1_tracking_result.mat' 
    path_to_trackability = os.path.join(base_dir, input_file_directory_trackability) + 'trackInROI/Channel_1_tracking_result_Trackability.mat'  
else:
    path_to_tracks = os.path.join(base_dir, input_file_directory) + 'Channel_1_tracking_result.mat'


tracks_dict = loadmat(path_to_tracks)
if import_trackability:
    trackability_dict = loadmat(path_to_trackability)

In [ ]:
# Get the keys of the dictionary

# tracks_dict.keys()
# if import_trackability:
    # trackability_dict.keys()

In [ ]:
# Get the key for your detections
key = 'tracksFinal'  # Replace with your actual key name
if import_trackability:
    key_trackability = 'trackabilityData'  # Replace with your actual key name

In [ ]:
def convert_tracks_to_dataframe(tracks_dict, key):
    """
    Convert MATLAB tracking data to a pandas DataFrame.
    
    Parameters:
    tracks_dict : dict
        Dictionary containing the MATLAB data (loaded using scipy.io.loadmat)
    key : str
        Key for the specific tracking data in the dictionary
    
    Returns:
    pd.DataFrame: DataFrame with columns [frame, mu_x, mu_y, mu_z, track_id]
    """
    import numpy as np
    import pandas as pd
    import scipy.io as sio
    
    # # Load the MATLAB file
    # mat_data = sio.loadmat(filepath, squeeze_me=False, struct_as_record=False)
    
    # Extract the tracks structure
    tracks = tracks_dict[key].flatten()
    
    # Create an empty list to store all track data
    all_tracks = []
    regular_tracks = []
    split_merge_tracks = []
    
    # Process each track
    for track_id in range(len(tracks)):

        # Handle numpy.void objects by accessing elements with field names
        track = tracks[track_id]
        # Access fields using dictionary-like indexing for numpy.void objects
        track_coords = track['tracksCoordAmpCG']
        seq_events = track['seqOfEvents']
        
        # Handle different shapes of seqOfEvents
        if seq_events.size == 0:
            continue  # Skip empty tracks
            
        # Reshape if necessary to ensure consistent format
        if len(seq_events.shape) == 1:
            seq_events = seq_events.reshape(1, -1)

        if np.isnan(seq_events[:, -1]).all():

            track_coords = track_coords[0]

            # save the track as a regular track
            regular_tracks.append(track_id)

            # Find start and end frames
            start_frame = int(seq_events[0, 0])
            end_frame = int(seq_events[-1, 0])

            # Determine the total number of frames from the size of tracksCoordAmpCG
            # Each frame has 8 columns [x y z a dx dy dz da]
            num_cols = track_coords.shape[0]
            num_frames = num_cols // 8

            # For each frame in the track's lifespan
            for frame_idx in range(num_frames):
                frame_number = start_frame + frame_idx
                # Extract x, y, z coordinates for current frame
                col_idx = frame_idx * 8

                x = track_coords[col_idx]
                y = track_coords[col_idx + 1]
                z = track_coords[col_idx + 2]
                amplitude = track_coords[col_idx + 3]

                track_data = {
                    'frame': frame_number,
                    'mu_x': round(x) if not np.isnan(x) else None,
                    'mu_y': round(y) if not np.isnan(y) else None,
                    'mu_z': round(z) if not np.isnan(z) else None,
                    'amplitude': amplitude,
                    'track_id': track_id  
                }

                all_tracks.append(track_data)

        else:
            # save the track as a split/merge track
            split_merge_tracks.append(track_id)

            # Determine if the track is split or merged (this seems a bit rudimentary, probably could be better)
            # Abhishek Raghunathan, 04/14/25
            if not np.isnan(seq_events[1, -1]): # This is assuming that we have only a single split event, it will fail otherwise.
                # Also that all split events are position 1 in seq_events
                track_flag = 'split'
                # print(f'Track {track_id} is split.')

            if not np.isnan(seq_events[2, -1]): # This is assuming that we have only a single merge event, it will fail otherwise.
                # Also that all merge events are position 2 in seq_events
                track_flag = 'merge'
                # print(f'Track {track_id} is merged.')
            

            # Process each segment in the track
            segments = np.unique(seq_events[:, 2]).astype(int)
            segments_min = np.min(segments) # Assuming the lowest segment ID is the first one (might not be true).
            start_frame_original = [] #To store the original start frame
            
            for segment_id in segments:
                # Find events related to this segment
                segment_events = seq_events[seq_events[:, 2] == segment_id]
                
                # Get start and end frames for this segment
                start_events = segment_events[segment_events[:, 1] == 1]
                end_events = segment_events[segment_events[:, 1] == 2]

                if segment_id == segments_min: # Assuming the lowest segment ID has the track which started first (lower value of first frame). CHECK THIS.
                    start_frame_original.append(int(start_events[0,0])) # This is the original start frame for the split or merge track
                    # print(start_frame_original)
                
                if start_events.size > 0 and end_events.size > 0:
                    start_frame = int(start_events[0, 0])
                    end_frame = int(end_events[0, 0])
                    
                    # Get row index for this segment (0-indexed)
                    segment_idx = segment_id - 1

                    # Get the row data for this segment
                    if segment_idx < len(track_coords):
                        segment_data = track_coords[segment_idx]
                        
                        # Calculate number of frames in this segment
                        segment_frames = end_frame - start_frame + 1
                        
                        # Process each frame in this segment
                        for frame_offset in range(segment_frames):
                            frame_number = start_frame + frame_offset

                            if (track_flag == 'split') or (track_flag == 'merge'): # This flag is unnecessary here, have it for legacy reasons.
                                col_idx = (start_frame + frame_offset - start_frame_original[0]) * 8 # This will handle cases where start_frame_original is not 1
                            else:
                                col_idx = frame_offset * 8
                            
                            # if track_id == 4370:
                            #     print(col_idx)
                            
                            
                            # Check if indices are within bounds
                            if col_idx + 3 < len(segment_data):
                                x = segment_data[col_idx]
                                y = segment_data[col_idx + 1]
                                z = segment_data[col_idx + 2]
                                amplitude = segment_data[col_idx + 3]
                                
                                track_data = {
                                    'frame': frame_number,
                                    'mu_x': round(x) if not np.isnan(x) else None,
                                    'mu_y': round(y) if not np.isnan(y) else None,
                                    'mu_z': round(z) if not np.isnan(z) else None,
                                    'amplitude': amplitude,
                                    'track_id': track_id,
                                    'segment_id': segment_id
                                }
                                all_tracks.append(track_data)
        
    
    track_df = pd.DataFrame(all_tracks)
    
    # Sort by track_id and frame
    track_df = track_df.sort_values(['track_id', 'frame'])

    return track_df, regular_tracks, split_merge_tracks

In [ ]:
df, regular_tracks, split_merge_tracks = convert_tracks_to_dataframe(tracks_dict, key)

In [ ]:
# # Diagnostic code to find the right path to trackability data
# def find_trackability_path(trackability_dict, key_trackability):
#     """
#     Navigate through the MATLAB structure to find the actual trackability data
#     """
#     print("=== FINDING TRACKABILITY DATA PATH ===\n")
    
#     trackability_data = trackability_dict[key_trackability]
#     print(f"1. trackability_data shape: {trackability_data.shape}")
#     print(f"   trackability_data type: {type(trackability_data)}")
    
#     # Level 1: trackability_data[0]
#     level1 = trackability_data[0]
#     print(f"\n2. trackability_data[0] shape: {level1.shape}")
#     print(f"   trackability_data[0] type: {type(level1)}")
    
#     # Level 2: trackability_data[0][0]  
#     level2 = level1[0]
#     print(f"\n3. trackability_data[0][0] shape: {level2.shape}")
#     print(f"   trackability_data[0][0] type: {type(level2)}")
#     print(f"   trackability_data[0][0] dtype: {level2.dtype}")
    
#     # Check if it has named fields
#     if hasattr(level2.dtype, 'names') and level2.dtype.names:
#         print(f"   Field names: {level2.dtype.names}")
        
#         # Look for segTrackability field
#         if 'segTrackability' in level2.dtype.names:
#             seg_track = level2['segTrackability']
#             print(f"\n4. Found segTrackability field!")
#             print(f"   segTrackability shape: {seg_track.shape}")
#             print(f"   segTrackability type: {type(seg_track)}")
            
#             # Try to access the actual data
#             if seg_track.shape == (1,):
#                 actual_data = seg_track[0]
#                 print(f"\n5. segTrackability[0] shape: {actual_data.shape}")
#                 print(f"   segTrackability[0] type: {type(actual_data)}")
                
#                 # Show first few entries
#                 print(f"\n6. First few trackability arrays:")
#                 for i in range(min(3, len(actual_data))):
#                     print(f"   Track {i}: {actual_data[i].flatten()[:10]}...")
                    
#                 return actual_data
#             else:
#                 print(f"\n5. segTrackability data directly accessible")
#                 return seg_track
#         else:
#             print("   segTrackability field not found in named fields")
            
#             # Try to access the data directly
#             if level2.shape == (1,):
#                 level3 = level2[0]
#                 print(f"\n4. trackability_data[0][0][0] shape: {level3.shape}")
#                 print(f"   trackability_data[0][0][0] type: {type(level3)}")
                
#                 # If it's an array, try to access elements
#                 if len(level3) > 0:
#                     print(f"\n5. First element: {level3[0]}")
#                     if hasattr(level3[0], 'shape'):
#                         print(f"   First element shape: {level3[0].shape}")
                        
#                 return level3
#     else:
#         print("   No named fields, trying direct access")
#         if level2.shape == (1,):
#             level3 = level2[0]
#             print(f"\n4. trackability_data[0][0][0] shape: {level3.shape}")
#             print(f"   trackability_data[0][0][0] type: {type(level3)}")
#             return level3
#         else:
#             return level2

# # Run the diagnostic
# actual_trackability_data = find_trackability_path(trackability_dict, key_trackability)

In [ ]:
import pandas as pd
import numpy as np

def convert_trackability_to_dataframe(trackability_dict, key_trackability):
    """
    Convert MATLAB trackability data to a pandas DataFrame.
    
    Parameters:
    trackability_dict : dict
        Dictionary containing the MATLAB trackability data (loaded using scipy.io.loadmat)
    key_trackability : str
        Key for the specific trackability data in the dictionary
    
    Returns:
    pd.DataFrame: DataFrame with columns [track_id, segTrackability]
    """
    # Extract the trackability data
    trackability_data = trackability_dict[key_trackability]
    
    # Navigate to the segTrackability field: trackability_data[0][0]['segTrackability']
    seg_trackability = trackability_data[0][0]['segTrackability']  # Shape: (1, 252)
    
    # Create list to store all trackability records
    all_trackability = []
    
    # Loop through each track (252 tracks in the columns)
    for track_idx in range(seg_trackability.shape[1]):
        try:
            # Extract trackability values for this track
            track_values = seg_trackability[0, track_idx]  # Get the track data from column track_idx
            
            # Convert to list, flattening if needed
            if isinstance(track_values, np.ndarray):
                if track_values.ndim > 1:
                    # Flatten multi-dimensional arrays
                    trackability_list = track_values.flatten().tolist()
                else:
                    trackability_list = track_values.tolist()
            else:
                trackability_list = [track_values] if not isinstance(track_values, list) else track_values
            
            # Create record for this track
            track_record = {
                'track_id': track_idx,
                'segTrackability': trackability_list
            }
            
            all_trackability.append(track_record)
            
        except Exception as e:
            print(f"Error processing track {track_idx}: {e}")
            # Add empty record for this track to maintain indexing
            all_trackability.append({
                'track_id': track_idx,
                'segTrackability': []
            })
            continue
    
    # Create DataFrame
    df = pd.DataFrame(all_trackability).set_index('track_id')
    
    # Print summary
    if not df.empty:
        total_tracks = len(df)
        non_empty_tracks = sum(1 for track_list in df['segTrackability'] if len(track_list) > 0)
        print(f"Created DataFrame with {total_tracks} tracks")
        print(f"Number of tracks with trackability data: {non_empty_tracks}")
        
        # Show some sample data
        if non_empty_tracks > 0:
            print("\nSample trackability data:")
            for idx, (track_id, row) in enumerate(df.iterrows()):
                if len(row['segTrackability']) > 0:
                    values = row['segTrackability']
                    print(f"Track {track_id}: {len(values)} values - {values[:5]}{'...' if len(values) > 5 else ''}")
                    if idx >= 2:  # Show only first 3 non-empty tracks
                        break
    else:
        print("No trackability data was found")
    
    return df

if import_trackability:
    df_trackability = convert_trackability_to_dataframe(trackability_dict, 'trackabilityData')

In [ ]:
if import_trackability:
    # Adding trackability segments to the tracks DataFrame
    # Adding NA to the beginning of each trackability list, since the first assocation of a track does not have a trackability score
    df_with_na = df_trackability.copy()
    # .apply lambda perform the operation on each element of the 'segTrackability' column
    # If the element is a list, it adds np.nan at the beginning, otherwise it creates a list with np.nan and the element
    df_with_na['segTrackability'] = df_with_na['segTrackability'].apply(
        lambda x: x + [np.nan] if isinstance(x, list) else [x] + [np.nan]
    )
    # Exploding the 'segTrackability' column to create a new row for each trackability value
    df_with_na = df_with_na.reset_index().explode('segTrackability')

    #### This is extremely dangerous, since it will not work if the trackability data is not in the same order as the tracks ####
    #### Howeever, since frame information is not available in the trackability data, we will have to assume that the trackability data is in the same order as the tracks ####
    #### Make sure to check the order of the trackability data before using this code ####
    df['segTrackability'] = df_with_na['segTrackability'].values

In [ ]:
# # Save df in path_to_tracks as a pickle file
output_path = path_to_tracks.replace('.mat', '.pkl')
print(f"Saving DataFrame to {output_path}")
df.to_pickle(output_path)

In [ ]:
len(split_merge_tracks)
# split_merge_tracks

In [ ]:
df
# df[(df['track_id'] == 37) & (df['segment_id'] == 1.0)]

In [ ]:
####This code supports the idea that the last track segment should have trackability score of NaN
#### rather than the first segment.
if import_trackability:
    # for each track_id in df, count the number of NaN values in the segTrackability column
    nan_counts = df.groupby('track_id')['segTrackability'].apply(lambda x: x.isna().sum())
    # Create a DataFrame from the counts
    nan_counts_df = nan_counts.reset_index()
    nan_counts_df.columns = ['track_id', 'nan_count']
    set(nan_counts_df['nan_count'])  # This will show the unique counts of NaN values per track_id
    # Find track ids with more than 1 NaN value in segTrackability
    tracks_with_multiple_nans = nan_counts_df[nan_counts_df['nan_count'] > 1]['track_id'].tolist()
    print(f"Tracks with multiple NaN values in segTrackability: {tracks_with_multiple_nans}")
# df[df['track_id'] == 5]

In [ ]:
# tracks = tracks_dict[key].flatten()
# track = tracks[37]
# track_coords = track['tracksCoordAmpCG']
# seq_events = track['seqOfEvents']
# seq_events

In [ ]:
# track_coords[0]

In [ ]:
# track_df = df[df['track_id'] == 33789]
# #sort based on segment id values
# track_df.sort_values(['segment_id', 'frame'])

In [ ]:
# # get non NaN values in track_coords[1]
# track_coords[1][~np.isnan(track_coords[1])]

In [ ]:
# # Subset rows of df with NaNs in any of mu_x, mu_y, mu_z
# df_nan = df[df[['mu_x', 'mu_y', 'mu_z']].isnull().any(axis=1)]
# # Get the track IDs of these rows
# nan_track_ids = df_nan['track_id'].unique()
# # Count the number of times each track ID appears in df_nan
# track_id_counts = df_nan['track_id'].value_counts()
# # Filter track IDs that appear more than once
# track_id_counts = track_id_counts[track_id_counts > 1]



In [ ]:
# track_id_counts